<a href="https://colab.research.google.com/github/jayfordamancio-debug/FUNDAI-Laboratories-AMANCIO/blob/main/Lab3_Game_AI_Amancio.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lab 3: Game AI Minimax with Alpha-Beta Pruning

## Fundamentals of Artificial Intelligence

**Name** Jayford L Amancio
**Course:** BSCS-AI
**Section:** 09282-FUNDAI
**Date:** September 1 2026

**Selected Game:** Tic-Tac-Toe
**GitHub URL:** https://github.com/jayfordamancio-debug

##Description
This laboratory implements a Tic-Tac-Toe AI using Minimax with Alpha-Beta pruning.
The game is playable inside Google Collab using  ipywidgets.

In [1]:

import math
import ipywidgets as widgets
from IPython.display import display


class TicTacToeGame:
    X = "X"
    O = "0"
    EMPTY = " "

    def __init__(self):
        self.board = [self.EMPTY] * 9
        self.current_player = self.X

    def available_moves(self):
        return [i for i, value in enumerate(self.board) if value == self.EMPTY]

    def make_move(self, move):
        self.board[move] = self.current_player
        self.current_player = self.O if self.current_player == self.X else self.X

    def undo_move(self, move):
        self.board[move] = self.EMPTY
        self.current_player = self.O if self.current_player == self.X else self.X

    def get_winner(self):
        winning_lines = [
            (0, 1, 2),
            (3, 4, 5),
            (6, 7, 8),
            (0, 3, 6),
            (1, 4, 7),
            (2, 5, 8),
            (0, 4, 8),
            (2, 4, 6)
        ]

        for a, b, c in winning_lines:
            if self.board[a] != self.EMPTY and self.board[a] == self.board[b] == self.board[c]:
                return self.board[a]

        return None

    def is_draw(self):
        return self.get_winner() is None and len(self.available_moves()) == 0

    def is_terminal(self):
        return self.get_winner() is not None or len(self.available_moves()) == 0

    def utility(self):
        winner = self.get_winner()

        if winner == self.X:
            return 1
        elif winner == self.O:
            return -1
        else:
            return 0


def minimax_alpha_beta(game, alpha=-math.inf, beta=math.inf):
    if game.is_terminal():
        return game.utility(), None

    # MAX player: X
    if game.current_player == TicTacToeGame.X:
        best_value = -math.inf
        best_move = None

        for move in game.available_moves():
            game.make_move(move)
            value, _ = minimax_alpha_beta(game, alpha, beta)
            game.undo_move(move)

            if value > best_value:
                best_value = value
                best_move = move

            alpha = max(alpha, best_value)

            if alpha >= beta:
                break

        return best_value, best_move

    # MIN player: 0
    else:
        best_value = math.inf
        best_move = None

        for move in game.available_moves():
            game.make_move(move)
            value, _ = minimax_alpha_beta(game, alpha, beta)
            game.undo_move(move)

            if value < best_value:
                best_value = value
                best_move = move

            beta = min(beta, best_value)

            if alpha >= beta:
                break

        return best_value, best_move


class TicTacToeUI:
    def __init__(self):
        self.game = TicTacToeGame()

        self.buttons = [
            widgets.Button(
                description=" ",
                layout=widgets.Layout(width="60px", height="60px")
            )
            for i in range(9)
        ]

        for i in range(9):
            self.buttons[i].on_click(lambda btn, idx=i: self.on_cell_click(idx))

        self.status = widgets.HTML(value="<b>Human X moves first.</b>")

        self.reset_button = widgets.Button(
            description="Reset",
            button_style="info"
        )

        self.reset_button.on_click(self.on_reset)

        self.grid = widgets.GridBox(
            children=self.buttons,
            layout=widgets.Layout(
                grid_template_columns="repeat(3, 60px)",
                grid_gap="5px"
            )
        )

        self.widget = widgets.VBox([
            self.status,
            self.grid,
            self.reset_button
        ])

        display(self.widget)

        self.refresh()

    def refresh(self):
        for i, button in enumerate(self.buttons):
            button.description = self.game.board[i]
            button.disabled = self.game.is_terminal() or self.game.board[i] != TicTacToeGame.EMPTY

        if self.game.is_terminal():
            winner = self.game.get_winner()

            if winner:
                self.status.value = f"<b>Player {winner} wins!</b>"
            else:
                self.status.value = "<b>Draw!</b>"
        else:
            self.status.value = f"<b>Current player: {self.game.current_player}</b>"

    def on_cell_click(self, index):
        if self.game.board[index] != TicTacToeGame.EMPTY:
            return

        if self.game.is_terminal():
            return

        if self.game.current_player != TicTacToeGame.X:
            return

        # Human move
        self.game.make_move(index)

        # AI move if game is not finished
        if not self.game.is_terminal():
            ai_move = self.get_ai_move()

            if ai_move is not None:
                self.game.make_move(ai_move)

        self.refresh()

    def get_ai_move(self):
        _, move = minimax_alpha_beta(self.game)
        return move

    def on_reset(self, button):
        self.game = TicTacToeGame()
        self.refresh()


TicTacToeUI()







## Algorithm Explanation

###Minimax
The Ai evulates possible future game states.
Player X maximizes utility, while Player O minimizes utility.

### Alpha-Beta Pruning
Alpha- beta pruning removes bracnhes that cannot chnage the final decision.
This makes the ai faster while preserving the optimal result.

## Utility
- x wins +1
- Draw: 0
- O wins -1

## Guide Questions and Answers

### 1. Which player does the AI control?
**Answer:**   Player 2 which is the X

### 2. What utility values were used?
**Answer: ** x wins +1
Draw: 0
O wins -1


### 3. How does Alpha-Beta pruning improve Minimax?
**Answer:** Alpha-beta pruning improves Minimax by making it faster without changing the final decision.

### 4. What happends when the human chooses a move that leads to draw
**Answer:** Then the UI displays its a "Draw"

### 5. Why is Tic-Tac-Toe suitable for full Minimax search?
**Answer:** because its game tree is small enough for a computer to explore essentially all possible future moves.

## Reflection

### Challenges Encountered
My Challenges encoutered about this code is my indentions were not good and also some of the letters good and also the spacings

## What I Learned
I learned how Minimax helps AI choose the best move using utility values. I also learned that alpha-beta pruning makes Minimax faster by skipping unnecessary moves.